In [17]:
# install importance library
!pip install -q ultralytics opencv-python matplotlib scipy pandas tqdm pyyaml

In [22]:
import numpy as np
import pandas as pd
import os
import shutil
import torch

In [18]:
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name:    Tesla T4


In [6]:
if os.path.exists("Low-Light-Image-Enhancement-and-Downstream-Recognition"):
    shutil.rmtree("Low-Light-Image-Enhancement-and-Downstream-Recognition")

!git clone https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition

Cloning into 'Low-Light-Image-Enhancement-and-Downstream-Recognition'...
remote: Enumerating objects: 14774, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 14774 (delta 5), reused 19 (delta 3), pack-reused 14747 (from 1)
Receiving objects: 100% (14774/14774), 269.16 MiB | 19.37 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (14713/14713), done.


In [14]:
os.chdir('/content/Low-Light-Image-Enhancement-and-Downstream-Recognition')

!ls

Dataset  Doc  Notebooks  README.md  requirements.txt  run.py  src


# Chose model to run

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
print("Vị trí hiện tại (CWD):", os.getcwd())

Vị trí hiện tại (CWD): /content


# Module Edge Loss use pytorch

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class SobelEdgeLoss(nn.Module):
    def __init__(self, device='cuda'):
        super(SobelEdgeLoss, self).__init__()
        # Kernel Sobel phương ngang và dọc[cite: 2]
        sobel_x = torch.tensor([[-1., 0., 1.],
                                [-2., 0., 2.],
                                [-1., 0., 1.]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

        sobel_y = torch.tensor([[-1., -2., -1.],
                                [ 0.,  0.,  0.],
                                [ 1.,  2.,  1.]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

        self.register_buffer('sobel_x', sobel_x.to(device))
        self.register_buffer('sobel_y', sobel_y.to(device))
        self.loss_fn = nn.L1Loss()

    def get_edges(self, tensor_img):
        # Ép về 1 kênh để tính gradient không gian
        if tensor_img.shape[1] > 1:
            tensor_img = torch.mean(tensor_img, dim=1, keepdim=True)
        grad_x = F.conv2d(tensor_img, self.sobel_x, padding=1)
        grad_y = F.conv2d(tensor_img, self.sobel_y, padding=1)
        return torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-6)

    def forward(self, feature_map, input_img):
        # 1. Lấy biên từ Feature Map tầng nông của YOLOv8
        pred_edge = self.get_edges(feature_map)

        # 2. Resize ảnh gốc tối về cùng spatial size với feature map
        if pred_edge.shape[-2:] != input_img.shape[-2:]:
            input_img = F.interpolate(input_img, size=pred_edge.shape[-2:], mode='bilinear', align_corners=False)

        target_edge = self.get_edges(input_img)
        return self.loss_fn(pred_edge, target_edge)

# Model train

In [ ]:
# Khởi tạo loss
edge_criterion = EdgeAwareLoss(device=device)
lambda_edge = 0.05  # Trọng số cân bằng

model.train()
for batch, (images, targets) in enumerate(dataloader):
    images = images.to(device)

    # 1. Forward pass: Lấy cả đặc trưng tầng nông và output dự đoán
    # (Giả sử model của em trả về features từ lớp Conv đầu và logits dự đoán)
    shallow_features, detections = model(images)

    # 2. Tính toán các hàm Loss
    loss_det = compute_detection_loss(detections, targets) # Classification + Box Loss
    loss_edge = edge_criterion(shallow_features, images)

    # Tổng loss kết hợp
    total_loss = loss_det + lambda_edge * loss_edge

    # 3. Lan truyền ngược và tối ưu tham số [source: 6, 7]
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()